In [1]:
# if needed : %pip install 'git+https://github.com/MydonSolutions/seticore#egg=seticore&subdirectory=python'
import pandas as pd
from astropy.time import Time
import os
import capnp
import glob
import multiprocessing
from collections import defaultdict
from scipy.spatial import cKDTree
import numpy as np
from seticore import viewer 
hit_capnp = viewer.hit_capnp

kj/filesystem-disk-unix.c++:1734: warning: PWD environment variable doesn't match current directory; pwd = /mnt_home/ellambishop


In [ ]:
"""
Preliminary file organization
------------------------------
Can skip if files are already organized 
"""

directory = "/datax/scratch/jaym/meerkat_data/"

# Get list of candidate files
folders = [f for f in os.listdir(directory) if f.startswith("blpn")]
full_paths = [os.path.join(directory, f) for f in folders]

hits_paths=[]
for path in full_paths:
    for subfolder in os.listdir(path):
        subfolder_path = os.path.join(path, subfolder)
        if os.path.isdir(subfolder_path):  
            for file in os.listdir(subfolder_path):
                if file.endswith(".hits"):
                   hits_paths.append(os.path.join(subfolder_path, file))

In [ ]:
"""  
Projecting MeerKAT columns to VLA data format such that RFI reduction pipeline can 
easily take data in the same format.
------------------------------------

"""

def align_meerkat_to_vla(meerkat_df):
    rename_map = {
        'signal_driftRate': 'signal_drift_rate',
        'signal_incoherentPower': 'signal_incoherent_power',
        'signal_numTimesteps': 'signal_num_timesteps',
        'filterbank_ra': 'ra_degrees',
        'filterbank_dec': 'dec_degrees',
        'filterbank_sourceName': 'source_name',
        'filterbank_tstart': 'tstart',
        'filterbank_beam': 'beam_id', 
        'filterbank_tsamp': 'filterbank_tsamp'
    }
    meerkat_df = meerkat_df.rename(columns=rename_map)

    # --- Handle RA/DEC conversion safely ---
    if 'ra_degrees' in meerkat_df.columns:
        meerkat_df['ra_hours'] = meerkat_df['ra_degrees'] * 15.0
    else:
        print(" No RA column found. Filling RA/DEC as NaN.")
        meerkat_df['ra_degrees'] = None
        meerkat_df['ra_hours'] = None

    if 'dec_degrees' not in meerkat_df.columns:
        meerkat_df['dec_degrees'] = None

    # --- Add placeholder columns missing in MeerKAT ---
    for col in ['file_uri']:
        if col not in meerkat_df.columns:
            meerkat_df[col] = None

    vla_columns = [
        'file_uri', 'source_name', 'beam_id',
        'ra_hours', 'dec_degrees', 'tstart',
        'signal_frequency', 'signal_beam', 'signal_drift_rate', 'signal_snr',
        'signal_power', 'signal_incoherent_power', 'signal_num_timesteps', 'filterbank_tsamp'
    ]

    # Add any missing columns as None
    for c in vla_columns:
        if c not in meerkat_df.columns:
            meerkat_df[c] = None

    return meerkat_df[vla_columns]


In [ ]:
def hits_to_df(hits, exclude_data=True):

    """   
    Load guppi MeerKAT hits files and convert them to pickle 
    files to then read in as dfs. 
    -----------------------------------------------------------
    Skip if format already desirable.
    """
    data = []
    for h in hits:
        entry = {}
        # fields is a dict with field names as keys
        for fname in h.schema.as_struct().fields.keys():
            value = getattr(h, fname)
            if hasattr(value, "schema"):  # nested struct
                
                # Get subfield names as dict keys as well
                subfields = value.schema.fields
                if isinstance(subfields, dict):
                    subfield_names = subfields.keys()
                else:
                    # fallback: list or other
                    try:
                        subfield_names = [sf.name for sf in subfields]
                    except Exception:
                        subfield_names = subfields

                for sfname in subfield_names:
                    if exclude_data and sfname == "data":
                        continue
                    entry[f"{fname}_{sfname}"] = getattr(value, sfname)
            else:
                entry[fname] = value
        data.append(entry)
    return pd.DataFrame(data)


pickle_dir = "/datax/scratch/ellambishop/meerkat_pkls/"
os.makedirs(pickle_dir, exist_ok=True)  # Create directory if it doesn't exist

for idx, hits_file in enumerate(hits_paths):
    with open(hits_file, "rb") as f:
        hits = list(hit_capnp.Hit.read_multiple(f))
    
    df = hits_to_df(hits)
    df = align_meerkat_to_vla(df)
    
    # Add observation ID column (filename of the .hits file)
    df['observation_id'] = os.path.basename(hits_file)
    # Create unique pickle filename
    base_name = os.path.splitext(os.path.basename(hits_file))[0]
    pickle_name = f"{idx:03d}_seticore_search_{base_name}.pkl"
    pickle_path = os.path.join(pickle_dir, pickle_name)

    df.to_pickle(pickle_path)
    print(f"Saved {pickle_path}")


In [ ]:
def convert_tstart_to_mjd(df, tstart_col='tstart'):
    mjd_list = []
    dt_list = []
    for t in df[tstart_col]:
        try:
            if t > 2.4e6 and t < 3e6:  # plausible JD
                time_obj = Time(t, format='jd')
            elif t > 1e6:  # large number, assume UNIX seconds
                time_obj = Time(t, format='unix')
            else:  # too small / unknown
                mjd_list.append(None)
                dt_list.append(None)
                continue

            mjd_list.append(time_obj.mjd)
            dt_list.append(time_obj.to_datetime())
        except Exception:
            mjd_list.append(None)
            dt_list.append(None)

    df['tstart'] = mjd_list
    return df

def stamps_to_df(stamp_paths):

    """  
    Load in stamp files and convert units for proper matching. Usefull for creating
    waterfall plots with cleaned stamp file data.

    """
    records = []
    for path in stamp_paths:
        these = list(viewer.read_stamps(path, find_recipe=True))
        observation_id = os.path.basename(path)

        for s in these:
            rec = {
                'file_uri': path,
                'beam_id':s.stamp.signal.beam,
                'observation_id': observation_id, 
                'tstart': s.stamp.tstart, 
                'filterbank_tsamp': s.stamp.tsamp,
                'source_name': s.stamp.sourceName, 
                'signal_frequency': s.stamp.signal.frequency,
                'signal_drift_rate': s.stamp.signal.driftRate,
                'signal_snr': s.stamp.signal.snr,
                'signal_power': s.stamp.signal.power,
                'signal_incoherent_power': s.stamp.signal.incoherentPower,
                'signal_num_timesteps': s.stamp.signal.numTimesteps,
                'signal_beam': s.stamp.signal.beam,
                'ra_hours': s.stamp.ra,
                'dec_degrees': s.stamp.dec
            }
          
            records.append(rec)
    return pd.DataFrame(records)
        
STAMPS_GLOB = '/datax/scratch/jaym/meerkat_data/*/seticore_search/*.stamps'
stamp_paths = glob.glob(STAMPS_GLOB, recursive=True)
stamp_df = stamps_to_df(stamp_paths)
stamp_df = convert_tstart_to_mjd(stamp_df)

stamp_df['ra_hours'] = stamp_df['ra_hours'] * 15.0



In [ ]:
""" 
Cell functions used to eliminate hits with RFI drift rate and snr threshold analysis
"""

intervals = pd.read_csv("RFI_band_files/nrao-meerkat-rfi.csv")
intervals.columns = intervals.columns.str.strip()

def parse_freq(freq_str):
    """ 
    Parse the 'Frequency (MHz)' column
    """
    if pd.isna(freq_str):
        return (np.nan, np.nan)
    freq_str = str(freq_str).replace(" ", "")
    if "-" in freq_str:
        parts = freq_str.split("-")
        return float(parts[0]), float(parts[1])
    else:
        val = float(freq_str)
        return val, val

intervals[['start_frequency', 'end_frequency']] = intervals['Frequency (MHz)'].apply(lambda x: pd.Series(parse_freq(x)))

# Drop rows that couldn’t be parsed
rfi_bands = intervals.dropna(subset=['start_frequency', 'end_frequency'])[['start_frequency', 'end_frequency']].values

incoherent_list = []
coherent_list = []

def filter_hits_by_rfi(df):
    freqs = df['signal_frequency'].values
    keep_mask = np.ones(len(freqs), dtype=bool)
    for low, high in rfi_bands:
        keep_mask &= ~((freqs >= low) & (freqs <= high))
    return df[keep_mask]

def reduction(df):
    # Filter out zero drift rate
    df = df[df['signal_drift_rate'] != 0].copy()
    # Keep only low-SNR AND low-drift
    df = df[(df['signal_snr'] <= 50) & (df['signal_drift_rate'] <= 4)].copy()
    # Tag coherent vs. incoherent
    df['is_gaia'] = df['source_name'].str.startswith("Gaia")
    gaia_df = df[df['is_gaia']].copy()
    named_df = df[~df['is_gaia']].copy()
    incoherent_list.append(named_df)
    coherent_list.append(gaia_df)

#Process files
files = glob.glob('/datax/scratch/ellambishop/meerkat_pkls/*.pkl')
print(f"Found {len(files)} pickle files.")

for f in files:
    df = pd.read_pickle(f)
    df = filter_hits_by_rfi(df)
    if not df.empty:
        reduction(df)


[[1000.   1000.  ]
 [1030.   1030.  ]
 [1025.   1150.  ]
 [1090.   1090.  ]
 [1166.   1186.  ]
 [1200.   1200.  ]
 [1222.5  1223.5 ]
 [1217.   1237.  ]
 [1231.   1231.  ]
 [1233.71 1233.71]]
Found 252 pickle files.


In [ ]:
"""  
Matching stamps files to hits in pkl files
"""

import pandas as pd

if coherent_list:
    df_gaia = pd.concat(coherent_list, ignore_index=True)
else:
    df_gaia = pd.DataFrame()

if incoherent_list:
    df_named = pd.concat(incoherent_list, ignore_index=True)
else:
    df_named = pd.DataFrame()


# Create a base observation ID for matching (everything before .hits/.stamps)
df_named['obs_base'] = df_named['observation_id'].str.split('.hits').str[0]
stamp_df['obs_base'] = stamp_df['observation_id'].str.split('.stamps').str[0]

df_named['matched_stamp_uris'] = [[] for _ in range(len(df_named))]

# Optional: define stamp frequency ranges if not already present
if 'freq_start' not in stamp_df or 'freq_end' not in stamp_df:
    stamp_df['freq_start'] = stamp_df['signal_frequency'] - 0.1e7
    stamp_df['freq_end']   = stamp_df['signal_frequency'] + 0.1e7

for hit_idx, hit in df_named.iterrows():
    stamps_in_obs = stamp_df[stamp_df['obs_base'] == hit['obs_base']]
    matched = stamps_in_obs[
        (stamps_in_obs['freq_start'] <= hit['signal_frequency']) &
        (stamps_in_obs['freq_end'] >= hit['signal_frequency'])
    ]
    df_named.at[hit_idx, 'matched_stamp_uris'] = matched['observation_id'].tolist()

# Track number of matches
df_named['num_matches'] = df_named['matched_stamp_uris'].apply(len)

# Check 
print(df_named[['source_name','ra_hours','dec_degrees','matched_stamp_uris','num_matches']].head())
print(f"Total hits with ≥1 matched stamp: {(df_named['num_matches'] > 0).sum()}")


  source_name   ra_hours  dec_degrees  \
0  J0042+1246  10.679417    12.782667   
1  J0042+1246  10.679417    12.782667   
2  J0042+1246  10.679417    12.782667   
3  J0042+1246  10.679417    12.782667   
4  J0042+1246  10.679417    12.782667   

                                  matched_stamp_uris  num_matches  
0  [guppi_60578_82556_000102_J0042+1246_0001.stam...           81  
1  [guppi_60578_83894_000108_J0042+1246_0001.stam...           83  
2  [guppi_60578_83894_000108_J0042+1246_0001.stam...           83  
3  [guppi_60579_00171_000118_J0042+1246_0001.stam...           89  
4  [guppi_60579_00171_000118_J0042+1246_0001.stam...           89  
Total hits with ≥1 matched stamp: 769


In [ ]:
"""  
Save results in pkl files
------------------------
Update naming schema if desired
"""
output_dir = "/datax/scratch/ellambishop/meerkat_hits"
os.makedirs(output_dir, exist_ok=True)


def save_df_if_not_empty(df, filename):
    if df is not None and not df.empty:
        out_path = os.path.join(output_dir, filename)
        df.to_pickle(out_path)
        print(f"Saved: {out_path}")
    else:
        print(f"Skipped saving {filename}: DataFrame empty or None")

save_df_if_not_empty(df_named, "meerkat_named.pkl")
save_df_if_not_empty(df_gaia, "meerkat_gaia.pkl")


Saved: /datax/scratch/ellambishop/meerkat_hits/meerkat_named.pkl
Saved: /datax/scratch/ellambishop/meerkat_hits/meerkat_gaia.pkl
